# 03 — Cohorte candidate d’un seul logement

L’objectif produit est d’estimer le prix d’une maison ou d’un appartement vendu seul. Le fichier DVF brut ne fournit pas ici d’identifiant documentaire exploitable. Le jeu dérivé public crée un `id_mutation` en détectant les changements consécutifs de date ou de prix. Cette convention est utile pour regrouper, mais ne prouve pas qu’un groupe corresponde exactement à un acte : des ventes voisines au même prix et à la même date peuvent être fusionnées.

Le script `ml/preprocessing/prepare_single_home_candidates.py` conserve les groupes contenant une seule ligne résidentielle distincte, sans autre type de local bâti que `Dépendance`, et répondant aux critères de base. Les lignes de terrain sans type local restent possibles dans le même groupe : le montant peut donc inclure du terrain. Le CSV initial reste inchangé.

In [ ]:
from pathlib import Path

import pandas as pd

BASE_PATH = Path("data/processed/dvf_eligible_rows_2025.csv")
CANDIDATE_PATH = Path(
    "data/processed/dvf_single_home_candidates_2025.csv"
)
for path in (BASE_PATH, CANDIDATE_PATH):
    if not path.exists():
        raise FileNotFoundError(
            f"{path} absent. Consulte data/README.md pour le créer."
        )

base = pd.read_csv(BASE_PATH, parse_dates=["sale_date"], low_memory=False)
candidates = pd.read_csv(
    CANDIDATE_PATH, parse_dates=["sale_date"], low_memory=False
)
print(f"Cohorte initiale : {len(base):,} lignes")
print(f"Cohorte candidate : {len(candidates):,} lignes")
print(f"Part conservée : {len(candidates) / len(base):.1%}")
display(candidates.head())

## Comparer les types et les prix

Les statistiques décrivent des lignes DVF retenues par la règle candidate. Elles ne prouvent pas à elles seules que le montant est le prix isolé du logement.

In [ ]:
counts = pd.concat(
    [
        base["property_type"].value_counts().rename("eligible_rows"),
        candidates["property_type"]
        .value_counts()
        .rename("single_home_candidates"),
    ],
    axis=1,
).fillna(0).astype("int64")
display(counts)

price_summary = pd.concat(
    {
        "Cohorte initiale": base.groupby("property_type")[
            "sale_price_eur"
        ].quantile([0.5, 0.95, 0.99]),
        "Cohorte candidate": candidates.groupby("property_type")[
            "sale_price_eur"
        ].quantile([0.5, 0.95, 0.99]),
    },
    names=["cohort", "property_type", "quantile"],
).unstack("cohort")
display(price_summary.round(0))

print("Dates :", candidates["sale_date"].min(), "à", candidates["sale_date"].max())
print("Pièces égales à zéro :", int(candidates["rooms"].eq(0).sum()))
print("Doublons stricts après projection :", int(candidates.duplicated().sum()))

## Interprétation et limites

- Le regroupement est la convention du producteur du jeu dérivé public, pas un identifiant stable garanti.
- Un groupe gardé contient une seule ligne Maison/Appartement et aucun autre local bâti, sauf dépendance. Des lignes de terrain peuvent encore partager le prix.
- Des ventes distinctes adjacentes au même prix et à la même date peuvent avoir été fusionnées puis exclues.
- La cohorte candidate est plus adaptée à l’objectif que la cohorte initiale, mais reste une approximation à déclarer dans le projet.

**Résultat :** 735 202 lignes candidates, environ 69,3 % de la cohorte initiale. La prochaine analyse compare le premier modèle sur cette cohorte et examine l’effet des prix extrêmes.